In [ ]:
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] += ':' + os.path.expanduser('~/.cargo/bin')
!pip install maturin
!rustc --version
!maturin --version

info: downloading installer
warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.
info: profile set to default
info: default host triple is x86_64-unknown-linux-gnu
info: syncing channel updates for stable-x86_64-unknown-linux-gnu
info: latest update on 2026-05-28 for version 1.96.0 (ac68faa20 2026-05-25)
info: downloading 6 components
        cargo downloading [#              ]   10.64 MiB (40.63 MiB/s, ETA: 0s)
        cargo downloading [#              ]   10.64 MiB (40.63 MiB/s, ETA: 0s)
        cargo downloading [#              ]   10.64 MiB (40.63 MiB/s, ETA: 0s)
        cargo downloading [#              ]   10.64 MiB (39.98 MiB/s, ETA: 0s)
        cargo downloading [#              ]   10.64 MiB (40.01 MiB/s, ETA: 0s)
        cargo downloading [#              ]   10.64 MiB (40.19 MiB/s, ETA:

In [ ]:
!cd /content && maturin new rust_engine --bindings pyo3
%cd /content/rust_engine
!pwd

  ✨ Done! New project created rust_engine
/content/rust_engine
/content/rust_engine


In [ ]:
!echo 'csv = "1.3"' >> Cargo.toml
!tail -5 Cargo.toml

crate-type = ["cdylib"]

[dependencies]
pyo3 = "0.28.3"
csv = "1.3"


In [ ]:
%%writefile src/lib.rs
use pyo3::prelude::*;
use std::collections::HashMap;
use std::fs::File;
use csv::ReaderBuilder;

#[pyfunction]
fn hello() -> PyResult<String> {
    Ok("Hello from Rust!".to_string())
}

#[pyfunction]
fn multiply(a: f64, b: f64) -> PyResult<f64> {
    Ok(a * b)
}

#[pyfunction]
fn compute_product_score(csv_path: String) -> PyResult<Vec<(String, f64, f64, f64)>> {
    let file = File::open(csv_path)
        .map_err(|e| PyErr::new::<pyo3::exceptions::PyIOError, _>(e.to_string()))?;
    let mut reader = ReaderBuilder::new().has_headers(true).from_reader(file);
    let mut prod_data: HashMap<String, (f64, f64, i32)> = HashMap::new();

    for result in reader.records() {
        let record = match result {
            Ok(r) => r,
            Err(e) => return Err(PyErr::new::<pyo3::exceptions::PyValueError, _>(e.to_string())),
        };
        let product = record[0].to_string();
        let price: f64 = record[1].parse().unwrap_or(0.0);
        let rating: f64 = record[2].parse().unwrap_or(0.0);

        let entry = prod_data.entry(product).or_insert((0.0, 0.0, 0));
        entry.0 += price;
        entry.1 += rating;
        entry.2 += 1;
    }

    let mut results = Vec::new();
    for (product, (sum_price, sum_rating, count)) in prod_data {
        let avg_price = sum_price / count as f64;
        let avg_rating = sum_rating / count as f64;
        let score = (avg_price / 1500.0) * 0.4 + (avg_rating / 5.0) * 0.6;
        results.push((product, avg_price, avg_rating, score));
    }
    results.sort_by(|a, b| b.3.partial_cmp(&a.3).unwrap());
    Ok(results)
}

#[pymodule]
fn rust_engine(_py: Python, m: &Bound<'_, PyModule>) -> PyResult<()> {
    m.add_function(wrap_pyfunction!(hello, m)?)?;
    m.add_function(wrap_pyfunction!(multiply, m)?)?;
    m.add_function(wrap_pyfunction!(compute_product_score, m)?)?;
    Ok(())
}

Overwriting src/lib.rs


In [ ]:
!maturin build --release

In [ ]:
import sys, subprocess, glob
wheel = glob.glob("/content/rust_engine/target/wheels/*.whl")[0]
subprocess.run([sys.executable, "-m", "pip", "install", wheel, "--force-reinstall"])

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def extract_rating(article):
    star_element = article.select_one("p.star-rating")
    if star_element:
        classes = star_element.get("class", [])
        if "One" in classes: return 1
        if "Two" in classes: return 2
        if "Three" in classes: return 3
        if "Four" in classes: return 4
        if "Five" in classes: return 5
    return 0

books = []
for page in range(1, 5):
    url = f"https://books.toscrape.com/catalogue/page-{page}.html"
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(response.text, "html.parser")

    for article in soup.select(".product_pod"):
        title = article.h3.a["title"]
        price = float(article.select_one(".price_color").text.replace("£", "").replace("Â", ""))
        rating = extract_rating(article)
        books.append({"product": title, "price": price, "rating": rating})
    time.sleep(0.5)

df = pd.DataFrame(books)
df.to_csv("books.csv", index=False)
print(f"✅ {len(df)} livres scrappés")
print("\nDistribution des notes :")
print(df["rating"].value_counts().sort_index())
print("\nAperçu :")
print(df.head())

In [ ]:
# Télécharger et installer Deno
!curl -fsSL https://deno.land/install.sh | sh

# Ajouter Deno au PATH
import os
os.environ['PATH'] += ':' + os.path.expanduser('~/.deno/bin')

# Vérifier l'installation
!deno --version

In [ ]:
!pip install denomagics

import denomagics
from denomagics import register_deno_magics
register_deno_magics()

print("✅ Deno et denomagics prêts")

In [ ]:
import json
import pandas as pd

# S'assurer que df_scores existe
if 'df_scores' not in dir():
    # Recharger les scores si nécessaire
    import rust_engine
    scores = rust_engine.compute_product_score("books.csv")
    df_scores = pd.DataFrame(scores, columns=["product", "avg_price", "avg_rating", "score"])
    df_scores = df_scores.sort_values("score", ascending=False)

# Sauvegarder les 10 premiers pour le dashboard
df_scores.head(10).to_dict(orient="records")
with open("/tmp/scores.json", "w") as f:
    json.dump(df_scores.head(10).to_dict(orient="records"), f)

print("✅ scores.json créé avec succès")
print(f"Premier produit : {df_scores.iloc[0]['product'][:50]}...")

In [ ]:
%%d
const data = JSON.parse(await Deno.readTextFile("/tmp/scores.json"));

// Calcul des métriques
const avgPrice = (data.reduce((a,b) => a + b.avg_price, 0) / data.length).toFixed(2);
const avgRating = (data.reduce((a,b) => a + b.avg_rating, 0) / data.length).toFixed(2);
const maxScore = Math.max(...data.map(d => d.score)).toFixed(4);
const totalProducts = data.length;

// Génération des lignes du tableau avec barres de progression
let rows = "";
for (let item of data) {
    const scorePercent = (item.score / maxScore) * 100;
    const ratingStars = "⭐".repeat(Math.round(item.avg_rating));
    rows += `<tr>
        <td><strong>${item.product.slice(0, 45)}${item.product.length > 45 ? "..." : ""}</strong></td>
        <td><div class="price">£${item.avg_price.toFixed(2)}</div></td>
        <td><div class="rating">${ratingStars} ${item.avg_rating.toFixed(1)}</div></td>
        <td>
            <div class="progress-bar">
                <div class="progress-fill" style="width: ${scorePercent}%;">
                    <span class="score-value">${item.score.toFixed(4)}</span>
                </div>
            </div>
         </td>
    </tr>`;
}

const html = `<!DOCTYPE html>
<html lang="fr">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Dashboard Portfolio | Rust + Python + Deno</title>
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: 'Inter', -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            padding: 40px 20px;
        }

        .container {
            max-width: 1400px;
            margin: 0 auto;
        }

        /* Header */
        .header {
            text-align: center;
            margin-bottom: 40px;
        }

        .header h1 {
            font-size: 2.5rem;
            color: white;
            text-shadow: 0 2px 10px rgba(0,0,0,0.2);
            margin-bottom: 10px;
        }

        .header p {
            color: rgba(255,255,255,0.9);
            font-size: 1.1rem;
        }

        .badge {
            display: inline-block;
            background: rgba(255,255,255,0.2);
            backdrop-filter: blur(10px);
            padding: 5px 15px;
            border-radius: 20px;
            font-size: 0.85rem;
            margin-top: 15px;
        }

        /* Cartes métriques */
        .metrics {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(250px, 1fr));
            gap: 20px;
            margin-bottom: 40px;
        }

        .metric-card {
            background: white;
            border-radius: 20px;
            padding: 25px;
            text-align: center;
            box-shadow: 0 10px 30px rgba(0,0,0,0.2);
            transition: transform 0.3s ease;
        }

        .metric-card:hover {
            transform: translateY(-5px);
        }

        .metric-icon {
            font-size: 2.5rem;
            margin-bottom: 10px;
        }

        .metric-value {
            font-size: 2rem;
            font-weight: bold;
            color: #2c7da0;
        }

        .metric-label {
            color: #666;
            margin-top: 5px;
        }

        /* Tableau */
        .card {
            background: white;
            border-radius: 24px;
            padding: 30px;
            box-shadow: 0 20px 40px rgba(0,0,0,0.15);
            overflow-x: auto;
        }

        .card h2 {
            margin-bottom: 20px;
            color: #1a73e8;
            display: flex;
            align-items: center;
            gap: 10px;
        }

        table {
            width: 100%;
            border-collapse: collapse;
        }

        th {
            text-align: left;
            padding: 15px 10px;
            background: linear-gradient(135deg, #2c7da0, #3a9bd5);
            color: white;
            font-weight: 600;
        }

        th:first-child { border-radius: 12px 0 0 0; }
        th:last-child { border-radius: 0 12px 0 0; }

        td {
            padding: 15px 10px;
            border-bottom: 1px solid #eef2f6;
        }

        .price {
            font-weight: 600;
            color: #2c7da0;
        }

        .rating {
            color: #f4b400;
            font-weight: 500;
        }

        /* Barre de progression */
        .progress-bar {
            background: #eef2f6;
            border-radius: 20px;
            height: 30px;
            overflow: hidden;
            position: relative;
            width: 100%;
        }

        .progress-fill {
            background: linear-gradient(90deg, #34a853, #1e8e3e);
            height: 100%;
            border-radius: 20px;
            display: flex;
            align-items: center;
            justify-content: flex-end;
            padding-right: 10px;
            transition: width 0.5s ease;
        }

        .score-value {
            color: white;
            font-weight: bold;
            font-size: 0.8rem;
        }

        /* Footer */
        .footer {
            text-align: center;
            margin-top: 30px;
            color: rgba(255,255,255,0.7);
            font-size: 0.85rem;
        }

        @media (max-width: 768px) {
            body { padding: 20px; }
            .metric-card { padding: 15px; }
            .metric-value { font-size: 1.5rem; }
            .header h1 { font-size: 1.8rem; }
        }
    </style>
</head>
<body>
<div class="container">
    <div class="header">
        <h1>📊 Rust + Python Pipeline</h1>
        <p>Scraping réel → Calcul haute performance → Dashboard interactif</p>
        <div class="badge">🚀 Portfolio Senior | Data & Automation Engineer</div>
    </div>

    <div class="metrics">
        <div class="metric-card">
            <div class="metric-icon">📚</div>
            <div class="metric-value">${totalProducts}</div>
            <div class="metric-label">Livres analysés</div>
        </div>
        <div class="metric-card">
            <div class="metric-icon">💰</div>
            <div class="metric-value">£${avgPrice}</div>
            <div class="metric-label">Prix moyen</div>
        </div>
        <div class="metric-card">
            <div class="metric-icon">⭐</div>
            <div class="metric-value">${avgRating}/5</div>
            <div class="metric-label">Note moyenne</div>
        </div>
        <div class="metric-card">
            <div class="metric-icon">🏆</div>
            <div class="metric-value">${maxScore}</div>
            <div class="metric-label">Score max</div>
        </div>
    </div>

    <div class="card">
        <h2>📈 Top produits - Classement par score</h2>
        <table>
            <thead>
                <tr>
                    <th>📖 Produit</th>
                    <th>💷 Prix moyen</th>
                    <th>⭐ Note</th>
                    <th>📊 Score (performance)</th>
                </tr>
            </thead>
            <tbody>
                ${rows}
            </tbody>
        </table>
    </div>

    <div class="footer">
        <p>🔧 Pipeline complet : Scraping (requests/BeautifulSoup) → Analyse (Rust via maturin) → Visualisation (Deno)</p>
        <p>📅 Généré le ${new Date().toLocaleString()} | Données réelles de books.toscrape.com</p>
    </div>
</div>
</body>
</html>`;

await Deno.writeTextFile("/content/rust_engine/dashboard_pro.html", html);
console.log("✅ dashboard_pro.html généré !");
console.log("👉 Ouvre-le dans l'explorateur (fichiers → rust_engine → dashboard_pro.html)");

In [ ]:
from openai import OpenAI
from google.colab import userdata
import pandas as pd

GROQ_API_KEY = userdata.get('GROQ_API_KEY')

if GROQ_API_KEY:
    client = OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")

    # Métriques avancées
    top_product = df_scores.iloc[0]['product']
    top_score = df_scores.iloc[0]['score']
    avg_price = df_scores['avg_price'].mean()
    avg_rating = df_scores['avg_rating'].mean()
    total_books = len(df_scores)
    median_price = df_scores['avg_price'].median()
    price_std = df_scores['avg_price'].std()
    rating_distribution = df_scores['avg_rating'].value_counts().sort_index().to_dict()
    score_range = df_scores['score'].max() - df_scores['score'].min()

    prompt = f"""
You are a senior data consultant and technical writer. Generate a **premium, captivating, and deeply analytical** README.md for a data portfolio project.

## Project Context
- **Source**: Real web scraping from books.toscrape.com (80 books)
- **Tech Stack**: Python (requests, BeautifulSoup, Pandas, Polars), Rust (maturin, csv crate), Deno, GROQ AI
- **Pipeline**: Scraping → Rust computation → Data analysis → Interactive dashboard → AI insights

## Key Metrics
- Total books analyzed: {total_books}
- Average price: £{avg_price:.2f} (median: £{median_price:.2f}, std: £{price_std:.2f})
- Average rating: {avg_rating:.2f}/5
- Rating distribution: {rating_distribution}
- Top product: "{top_product}" (score: {top_score:.4f})
- Score range: {score_range:.4f}

## README Requirements (must follow exactly)

### 1. Title & Badges
Start with a compelling title and technology badges (Python, Rust, Deno, GROQ, BeautifulSoup, Pandas).

### 2. Executive Summary (2-3 paragraphs)
- What this project does (real scraping + high‑performance computing)
- Why it matters (data‑driven insights for e‑commerce)
- Key business value

### 3. Architecture & Data Flow
Describe the pipeline with emojis:
🌐 Scraping → 🦀 Rust computation → 🐍 Python analysis → 🎨 Deno dashboard → 🤖 GROQ insights

### 4. Deep Analytical Insights (most important)
Generate **real, data‑driven insights** based on the metrics above. Include:
- Price vs. rating correlation
- Top performers and underperformers
- Distribution patterns
- Statistical observations (mean, median, std, range)

### 5. Technical Stack Deep Dive
For each tool, explain WHY it was chosen and its specific role in the pipeline.

### 6. Results & Key Findings
A dedicated section with bullet points of the most actionable findings.

### 7. How to Reproduce
Clear, step‑by‑step instructions (without code blocks) to run the pipeline.

### 8. Conclusion & Next Steps
Future improvements and potential business applications.

### 9. Author & Portfolio
Brief note about you (Data & Automation Engineer, 100% written communication).

## Style Guidelines
- **Professional, confident, and insightful**
- Use emojis sparingly for visual breaks
- No markdown tables (use lists or paragraphs instead)
- No fluff — every sentence must add value
- Include a "Key Insights" callout box with the top 3 findings
- Length: 800–1200 words equivalent

Generate the complete README.md now.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.4
    )

    readme_content = response.choices[0].message.content

    # Sauvegarder dans le bon dossier
    with open("/content/rust_engine/README.md", "w", encoding="utf-8") as f:
        f.write(readme_content)

    print("✅ README.md (version senior) généré avec succès")
    print("\n" + "="*60)
    print("📄 APERÇU DU README")
    print("="*60)
    print(readme_content[:1000])
    print("\n... (contenu complet dans le fichier)")
else:
    print("❌ GROQ_API_KEY non trouvé")

✅ README.md (version senior) généré avec succès

📄 APERÇU DU README
# 📚 E-commerce Book Analysis: Uncovering Hidden Insights with Real Web Scraping and AI-Driven Analytics
[![Python](https://img.shields.io/badge/Python-3776AB?style=for-the-badge&logo=python&logoColor=white)](https://www.python.org/)
[![Rust](https://img.shields.io/badge/Rust-000000?style=for-the-badge&logo=rust&logoColor=white)](https://www.rust-lang.org/)
[![Deno](https://img.shields.io/badge/Deno-000000?style=for-the-badge&logo=deno&logoColor=white)](https://deno.land/)
[![GROQ](https://img.shields.io/badge/GROQ-000000?style=for-the-badge&logo=groq&logoColor=white)](https://groq.com/)
[![BeautifulSoup](https://img.shields.io/badge/BeautifulSoup-000000?style=for-the-badge&logo=beautifulsoup&logoColor=white)](https://www.crummy.com/software/BeautifulSoup/)
[![Pandas](https://img.shields.io/badge/Pandas-150458?style=for-the-badge&logo=pandas&logoColor=white)](https://pandas.pydata.org/)

## Executive Summary
This projec